## HPC Model Quality Review

This section evaluates all model runs in `models/hpc_runs` and compares them consistently.

Flow:
1. Build one shared evaluation data plan (top cell below).
2. Run one evaluation cell per model folder (or reuse existing results if a model type is not supported by `load_checkpoint`).
3. Compare all available `eval_results.json` files with `compare_eval`.

### Helpers

In [108]:
from pathlib import Path
import json
import os
import numpy as np
import yaml
import pandas as pd
from evaluation.compare_models import PhasedModelComparator

try:
    from tabulate import tabulate as _tabulate
except Exception:
    _tabulate = None

PROJECT_ROOT = Path('/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data').resolve()
HPC_RUNS_DIR = PROJECT_ROOT / 'models' / 'hpc_runs'
HPC_RUNS_DIR_PHASED = PROJECT_ROOT / "models" / "hpc_runs" / "3-phase"
COMPARISON_OUTPUT_DIR = PROJECT_ROOT / 'evaluation_comparison_hpc_runs'
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

def _discover_model_dirs_in(base_dir: Path) -> list[Path]:
    model_dirs = []

    if not base_dir.exists():
        print(f"Directory does not exist: {base_dir}")
        return model_dirs

    for d in sorted(base_dir.iterdir()):
        if not d.is_dir() or d.name == "UserProfiles":
            continue

        if _ranked_eval_json(d).exists() or _aggregated_eval_json(d).exists():
            model_dirs.append(d)

    return model_dirs

_CANONICAL_METRIC_BY_KEY = {
    'reconstructionrmse': 'Reconstruction RMSE',
    'reconstructionmse': 'Reconstruction MSE',
    'reconstructionloglikelihood': 'Reconstruction Log-Likelihood',
    'reconstructionlog-likelihood': 'Reconstruction Log-Likelihood',
    'flexibilitydistance': 'Flexibility Distance',
    'maximummeandiscrepancy(mmd)': 'Maximum Mean Discrepancy (MMD)',
    'maximummeandiscrepancy': 'Maximum Mean Discrepancy (MMD)',
    'set-mmd(permutationinvariant)': 'Maximum Mean Discrepancy (MMD)',
    'setmmd(permutationinvariant)': 'Maximum Mean Discrepancy (MMD)',
    'set-mmd': 'Maximum Mean Discrepancy (MMD)',
    'acfdistributionmatch(ks)': 'ACF Distribution Match (KS)',
    'classicalstatistics': 'Classical Statistics',
    'tstr(trainonsynthetic,testonreal)': 'TSTR (Train on Synthetic, Test on Real)',
    'tstr': 'TSTR (Train on Synthetic, Test on Real)',
    'aggregatedprofiles(flexibilitydistance)': 'Aggregated Profiles (Flexibility Distance)',
    'aggregatedprofiles': 'Aggregated Profiles (Flexibility Distance)',
    'per-conditionevaluation': 'Per-Condition Evaluation',
    'perconditionevaluation': 'Per-Condition Evaluation',
    'perconditioneval': 'Per-Condition Evaluation',
    'visualevaluations': 'Visual Evaluations',
}


def _norm_metric_key(name: str) -> str:
    return ''.join(ch for ch in str(name).lower() if ch.isalnum() or ch in {'(', ')', '-', ','})


def _canonical_metric_name(name: str) -> str:
    key = _norm_metric_key(name)
    return _CANONICAL_METRIC_BY_KEY.get(key, str(name))


def _metric_goal(metric_name: str) -> str:
    if metric_name == 'TSTR (Train on Synthetic, Test on Real)':
        return 'target=1'
    if 'Log-Likelihood' in metric_name or metric_name == 'Visual Evaluations':
        return 'higher'
    return 'lower'


def _is_better(metric_name: str, model_value, baseline_value):
    try:
        m = float(model_value)
        b = float(baseline_value)
    except Exception:
        return None
    if not np.isfinite(m) or not np.isfinite(b):
        return None
    goal = _metric_goal(metric_name)
    if goal == 'higher':
        return m > b
    if goal == 'target=1':
        return abs(m - 1.0) < abs(b - 1.0)
    return m < b


def _fmt_val(v) -> str:
    if v is None:
        return '-'
    if isinstance(v, str):
        return v
    if isinstance(v, (int, float, np.floating)):
        return f'{float(v):.4f}'
    return str(v)


def _render_table(rows, headers):
    if _tabulate is not None:
        return _tabulate(rows, headers=headers, tablefmt='grid')
    matrix = [headers] + [[str(c) for c in r] for r in rows]
    widths = [max(len(row[i]) for row in matrix) for i in range(len(headers))]
    sep = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'
    out = [sep]
    out.append('| ' + ' | '.join(headers[i].ljust(widths[i]) for i in range(len(headers))) + ' |')
    out.append(sep)
    for row in rows:
        out.append('| ' + ' | '.join(str(row[i]).ljust(widths[i]) for i in range(len(headers))) + ' |')
    out.append(sep)
    return '\n'.join(out)


def _extract_metric_map(data: dict) -> dict:
    if not isinstance(data, dict):
        return {}

    if isinstance(data.get('standalone_metrics'), dict) and data['standalone_metrics']:
        metric_map = {}
        for metric, payload in data['standalone_metrics'].items():
            canonical = _canonical_metric_name(metric)
            if isinstance(payload, dict):
                metric_map[canonical] = payload.get('value', None)
            else:
                metric_map[canonical] = payload
        return metric_map

    metric_map = {}
    ignored = {'standalone_metrics', 'results', 'metadata', 'run_info'}
    for key, value in data.items():
        if key in ignored:
            continue
        canonical = _canonical_metric_name(key)
        if isinstance(value, (int, float, np.floating)):
            metric_map[canonical] = float(value)
        elif isinstance(value, str) and value.upper() == 'ERROR':
            metric_map[canonical] = 'ERROR'
        elif isinstance(value, dict) and 'value' in value:
            metric_map[canonical] = value.get('value')
    return metric_map


def _coerce_condition_scalar(value):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(float(value)):
        return float(value)
    if isinstance(value, dict):
        if isinstance(value.get('fd', None), (int, float, np.floating)):
            return float(value['fd'])
        diffs = value.get('differences', None)
        if isinstance(diffs, dict) and isinstance(diffs.get('overall_mean_abs_diff', None), (int, float, np.floating)):
            return float(diffs['overall_mean_abs_diff'])
        if isinstance(value.get('value', None), (int, float, np.floating)):
            return float(value['value'])
    return None


def _extract_per_condition_metric_map(model_dir: Path) -> dict:
    p = model_dir / 'ranked_phase_1_eval' / 'eval_results_condition.json'
    if not p.exists():
        return {}
    try:
        with open(p, 'r') as f:
            data = json.load(f)
    except Exception:
        return {}

    ignored = {'aggregated_profile_plots', 'feeder_mapping_path', 'min_samples_per_feeder', 'window_size', 'max_lag', 'min_samples_per_group', 'predictor_type'}
    out = {}
    for metric_name, by_condition in data.items():
        if metric_name in ignored or not isinstance(by_condition, dict):
            continue
        for condition_name, by_bucket in by_condition.items():
            if not isinstance(by_bucket, dict):
                continue
            for bucket, payload in by_bucket.items():
                scalar = _coerce_condition_scalar(payload)
                if scalar is None:
                    continue
                out[f'Per-Condition/{metric_name}/{condition_name}={bucket}'] = float(scalar)
    return out


def _load_training_config_used(model_dir: Path) -> tuple[dict, str]:
    cfg_path = model_dir / 'training_config_used.yaml'
    if not cfg_path.exists():
        return {}, 'missing'
    try:
        with open(cfg_path, 'r') as f:
            cfg = yaml.safe_load(f) or {}
        return cfg if isinstance(cfg, dict) else {}, str(cfg_path.relative_to(PROJECT_ROOT))
    except Exception:
        return {}, str(cfg_path.relative_to(PROJECT_ROOT))


def _flatten_tags(tags):
    if isinstance(tags, list):
        return ', '.join(str(t) for t in tags)
    return str(tags) if tags is not None else '-'


def _print_model_hyperparameters(model_dir: Path):
    cfg, cfg_src = _load_training_config_used(model_dir)
    if not cfg:
        print('Hyperparameters: training_config_used.yaml not found or unreadable.')
        return

    model = cfg.get('model', {}) if isinstance(cfg.get('model', {}), dict) else {}
    training = cfg.get('training', {}) if isinstance(cfg.get('training', {}), dict) else {}
    data = cfg.get('data', {}) if isinstance(cfg.get('data', {}), dict) else {}
    dataloader = data.get('dataloader', {}) if isinstance(data.get('dataloader', {}), dict) else {}
    window = data.get('window', {}) if isinstance(data.get('window', {}), dict) else {}
    prior = model.get('prior', {}) if isinstance(model.get('prior', {}), dict) else {}
    encoder = model.get('encoder', {}) if isinstance(model.get('encoder', {}), dict) else {}
    decoder = model.get('decoder', {}) if isinstance(model.get('decoder', {}), dict) else {}
    lr_scheduler = training.get('lr_scheduler', {}) if isinstance(training.get('lr_scheduler', {}), dict) else {}
    conditioning = cfg.get('conditioning', {}) if isinstance(cfg.get('conditioning', {}), dict) else {}

    items = [
        ('config', cfg_src),
        ('model.type', model.get('type')),
        ('model.latent_dim', model.get('latent_dim')),
        ('model.encoder.type', encoder.get('type')),
        ('model.decoder.type', decoder.get('type')),
        ('model.prior.type', prior.get('type')),
        ('model.prior.dictionary_size', prior.get('dictionary_size')),
        ('training.optimizer', training.get('optimizer')),
        ('training.learning_rate', training.get('learning_rate')),
        ('training.weight_decay', training.get('weight_decay')),
        ('training.num_epochs', training.get('num_epochs')),
        ('training.beta', training.get('beta')),
        ('training.weighted_loss', training.get('weighted_loss')),
        ('training.batch_size', dataloader.get('batch_size')),
        ('training.window.input_len', window.get('input_len')),
        ('training.window.stride', window.get('stride')),
        ('training.lr_scheduler', lr_scheduler.get('scheduler_type') if lr_scheduler.get('enabled') else 'disabled'),
        ('conditioning.tags', _flatten_tags(conditioning.get('tags'))),
    ]

    rows = [[k, str(v)] for k, v in items if v is not None]
    print('Hyperparameters used')
    print(_render_table(rows, headers=['Key', 'Value']))


def _candidate_baseline_dirs(prefix: str) -> list[Path]:
    out = []
    for d in sorted(HPC_RUNS_DIR.glob(f'{prefix}_*')):
        if d.is_dir():
            out.append(d)
    return out


def _pick_baseline_dir(prefix: str) -> Path | None:
    cands = _candidate_baseline_dirs(prefix)
    return cands[-1] if cands else None


MEAN_BASELINE_DIR = _pick_baseline_dir('AverageBaselineModel')
QUANTILE_BASELINE_DIR = _pick_baseline_dir('QuantileBaselineModel')


def _ranked_eval_json(model_dir: Path) -> Path:
    return model_dir / 'ranked_phase_1_eval' / 'eval_results.json'


def _ranked_condition_json(model_dir: Path) -> Path:
    return model_dir / 'ranked_phase_1_eval' / 'eval_results_condition.json'


def _aggregated_eval_json(model_dir: Path) -> Path:
    return model_dir / "aggregated_phase_eval" / "eval_results.json"


def _discover_model_dirs() -> list[Path]:
    model_dirs = []
    for d in sorted(HPC_RUNS_DIR.iterdir()):
        if not d.is_dir() or d.name == "UserProfiles":
            continue

        if _ranked_eval_json(d).exists() or _aggregated_eval_json(d).exists():
            model_dirs.append(d)

    return model_dirs

# 1-phase
MODEL_DIRS = _discover_model_dirs()
MODEL_RUNS = [d.name for d in MODEL_DIRS]

# 3-phase
MODEL_DIRS_PHASED = _discover_model_dirs_in(HPC_RUNS_DIR_PHASED)
MODEL_RUNS_PHASED = [d.name for d in MODEL_DIRS_PHASED]


print(f"Discovered 1-phase model runs ({len(MODEL_RUNS)}): {MODEL_RUNS}")
print(f"Discovered 3-phase model runs ({len(MODEL_RUNS_PHASED)}): {MODEL_RUNS_PHASED}")

def _pretty_name(run_name: str) -> str:
    if run_name.startswith('AutoRegressiveTransformerVAE_'):
        return 'ART-VAE'
    if run_name.startswith('DiffusionModel_'):
        return 'Diffusion'
    if run_name.startswith('CVAE_') and 'guide_cnn' in run_name:
        return 'CVAE-Guide-CNN'
    if run_name.startswith('CVAE_') and 'guide' in run_name:
        return 'CVAE-Guide'
    if run_name.startswith('CVAE_') and 'vamp' in run_name:
        return 'CVAE-Vamp'
    if run_name.startswith('CVAE_') and 'hierarcchy' in run_name:
        return 'CVAE-Hierarchy'
    if run_name.startswith('CVAE_'):
        return 'CVAE'
    if run_name.startswith('AverageBaselineModel_'):
        return 'Mean Baseline'
    if run_name.startswith('QuantileBaselineModel_'):
        return 'Quantile Baseline'
    return run_name


def _load_model_metrics_from_ranked_eval(model_dir: Path) -> dict:
    eval_path = _ranked_eval_json(model_dir)
    if not eval_path.exists():
        return {}
    with open(eval_path, 'r') as f:
        data = json.load(f)
    overall = _extract_metric_map(data)
    per_cond = _extract_per_condition_metric_map(model_dir)
    return {**overall, **per_cond}


def _bool_str(v):
    if v is None:
        return '-'
    return 'True' if bool(v) else 'False'


def _print_model_summary_with_two_baselines(model_name: str):
    model_dir = HPC_RUNS_DIR / model_name
    if not model_dir.exists():
        print(f'Model directory missing: {model_dir}')
        return None

    mean_metrics = _load_model_metrics_from_ranked_eval(MEAN_BASELINE_DIR) if MEAN_BASELINE_DIR is not None else {}
    quant_metrics = _load_model_metrics_from_ranked_eval(QUANTILE_BASELINE_DIR) if QUANTILE_BASELINE_DIR is not None else {}
    model_metrics = _load_model_metrics_from_ranked_eval(model_dir)

    print(f"\n=== {model_name} ===")
    _print_model_hyperparameters(model_dir)

    if not model_metrics:
        print(f'No metrics found in {_ranked_eval_json(model_dir)}')
        return None

    metric_names = sorted(model_metrics.keys())
    rows = []
    for metric in metric_names:
        mv = model_metrics.get(metric)
        bv_mean = mean_metrics.get(metric)
        bv_quant = quant_metrics.get(metric)

        better_mean = None if bv_mean is None else _is_better(metric, mv, bv_mean)
        better_quant = None if bv_quant is None else _is_better(metric, mv, bv_quant)

        rows.append([
            metric,
            _metric_goal(metric),
            _fmt_val(mv),
            _fmt_val(bv_mean),
            _fmt_val(bv_quant),
            _bool_str(better_mean),
            _bool_str(better_quant),
        ])

    print('\nEvaluation summary (model vs both baselines)')
    headers = [
        'Metric',
        'Goal',
        'Model',
        'Mean Baseline',
        'Quantile Baseline',
        'Better Than Mean',
        'Better Than Quantile',
    ]
    print(_render_table(rows, headers))
    return model_metrics


print('Helper setup complete.')
print(f'Project root: {PROJECT_ROOT}')
print(f'HPC runs folder: {HPC_RUNS_DIR}')
print(f'Discovered model runs ({len(MODEL_RUNS)}): {MODEL_RUNS}')
print(f'Mean baseline dir: {MEAN_BASELINE_DIR.name if MEAN_BASELINE_DIR else "MISSING"}')
print(f'Quantile baseline dir: {QUANTILE_BASELINE_DIR.name if QUANTILE_BASELINE_DIR else "MISSING"}')

Discovered 1-phase model runs (9): ['AutoRegressiveTransformerVAE_20260424_170101', 'AverageBaselineModel_20260425_192621', 'CVAE_20260424_163736', 'CVAE_20260424_164906_guide', 'CVAE_20260424_173354_guide_cnn', 'CVAE_20260424_174908_hierarcchy', 'CVAE_20260424_180137_vamp', 'DiffusionModel_20260424_171827', 'QuantileBaselineModel_20260425_193656']
Discovered 3-phase model runs (5): ['AutoRegressiveTransformerVAE_20260426_184212', 'AverageBaselineModel_20260426_185808', 'DiffusionModel_20260426_182935', 'QuantileBaselineModel_20260426_190605', 'SetGuideVAE_20260426_180021']
Helper setup complete.
Project root: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data
HPC runs folder: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/models/hpc_runs
Discovered model runs (9): ['AutoRegressiveTransformerVAE_20260424_170101', 'AverageBaselineModel_20260425_192621', 'CVAE_20260424_163736', 'CVAE_20260424_164906_guide', 'CVAE_20260424_173354_guide_cnn', 'CVAE

In [109]:
# Validate discovered runs and required ranked_phase_1 files.
print('Discovered non-baseline model runs:')
for i, run_name in enumerate(MODEL_RUNS, start=1):
    print(f'  {i}. {run_name}')

print('\nBaseline model folders:')
print('  Mean baseline:', MEAN_BASELINE_DIR if MEAN_BASELINE_DIR else 'MISSING')
print('  Quantile baseline:', QUANTILE_BASELINE_DIR if QUANTILE_BASELINE_DIR else 'MISSING')

for bname, bdir in [('Mean', MEAN_BASELINE_DIR), ('Quantile', QUANTILE_BASELINE_DIR)]:
    if bdir is None:
        continue
    eval_path = _ranked_eval_json(bdir)
    cond_path = _ranked_condition_json(bdir)
    print(f'  {bname} eval_results.json exists: {eval_path.exists()} ({eval_path})')
    print(f'  {bname} eval_results_condition.json exists: {cond_path.exists()} ({cond_path})')

Discovered non-baseline model runs:
  1. AutoRegressiveTransformerVAE_20260424_170101
  2. AverageBaselineModel_20260425_192621
  3. CVAE_20260424_163736
  4. CVAE_20260424_164906_guide
  5. CVAE_20260424_173354_guide_cnn
  6. CVAE_20260424_174908_hierarcchy
  7. CVAE_20260424_180137_vamp
  8. DiffusionModel_20260424_171827
  9. QuantileBaselineModel_20260425_193656

Baseline model folders:
  Mean baseline: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/models/hpc_runs/AverageBaselineModel_20260425_192621
  Quantile baseline: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/models/hpc_runs/QuantileBaselineModel_20260425_193656
  Mean eval_results.json exists: True (/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/models/hpc_runs/AverageBaselineModel_20260425_192621/ranked_phase_1_eval/eval_results.json)
  Mean eval_results_condition.json exists: True (/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/m

In [110]:
# Quick preview of first discovered model's ranked_phase_1 eval file.
if not MODEL_RUNS:
    raise RuntimeError('No model runs with ranked_phase_1_eval/eval_results.json were discovered.')

_first_model_dir = HPC_RUNS_DIR / MODEL_RUNS[0]
_first_eval = _ranked_eval_json(_first_model_dir)
with open(_first_eval, 'r') as f:
    _first_payload = json.load(f)

print(f'Preview model: {MODEL_RUNS[0]}')
print(f'eval file: {_first_eval}')
print(f'top-level keys: {list(_first_payload.keys())[:20]}')

Preview model: AutoRegressiveTransformerVAE_20260424_170101
eval file: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/models/hpc_runs/AutoRegressiveTransformerVAE_20260424_170101/ranked_phase_1_eval/eval_results.json
top-level keys: ['Reconstruction RMSE', 'Reconstruction Log-Likelihood', 'Flexibility Distance', 'ACF Distribution Match (KS)', 'Classical Statistics', 'TSTR (Train on Synthetic, Test on Real)', 'Aggregated Profiles (Flexibility Distance)', 'Per-Condition Evaluation', 'Set-MMD (Permutation Invariant)']


In [111]:
# Baseline metrics loaded from baseline model folders.
mean_baseline_metrics = _load_model_metrics_from_ranked_eval(MEAN_BASELINE_DIR) if MEAN_BASELINE_DIR else {}
quantile_baseline_metrics = _load_model_metrics_from_ranked_eval(QUANTILE_BASELINE_DIR) if QUANTILE_BASELINE_DIR else {}

print('Mean baseline metrics loaded:', len(mean_baseline_metrics))
print('Quantile baseline metrics loaded:', len(quantile_baseline_metrics))

if not mean_baseline_metrics:
    print('Warning: mean baseline ranked_phase_1_eval/eval_results(.condition).json not found or empty.')
if not quantile_baseline_metrics:
    print('Warning: quantile baseline ranked_phase_1_eval/eval_results(.condition).json not found or empty.')

Mean baseline metrics loaded: 48
Quantile baseline metrics loaded: 48


### Per-Model Evaluation Cells

Run the cells below one by one. Each cell tries to run eval+visualization via `load_checkpoint` and falls back to existing artifacts if the model type is unsupported there.

#### 1-phase models

In [33]:
# 1) AutoRegressiveTransformerVAE
result_model_1 = _print_model_summary_with_two_baselines(MODEL_RUNS[0])


=== AutoRegressiveTransformerVAE_20260424_170101 ===
Hyperparameters used
+-----------------------------+----------------------------------------------------------------------------------------+
| Key                         | Value                                                                                  |
+-----------------------------+----------------------------------------------------------------------------------------+
| config                      | models/hpc_runs/AutoRegressiveTransformerVAE_20260424_170101/training_config_used.yaml |
| model.type                  | art                                                                                    |
| model.latent_dim            | 64                                                                                     |
| model.encoder.type          | dictionary                                                                             |
| model.decoder.type          | dictionary                                    

In [21]:
# 2) CVAE Dictionary
result_model_2 = _print_model_summary_with_two_baselines(MODEL_RUNS[1])


=== CVAE_20260424_163736 ===
Hyperparameters used
+-----------------------------+-----------------------------------------------------------------------------+
| Key                         | Value                                                                       |
+-----------------------------+-----------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_20260424_163736/training_config_used.yaml              |
| model.type                  | cvae                                                                        |
| model.latent_dim            | 64                                                                          |
| model.encoder.type          | mlp                                                                         |
| model.decoder.type          | mlp                                                                         |
| model.prior.type            | standard_gaussian                    

In [22]:
# 3) CVAE Dictionary + Hierarchy
result_model_3 = _print_model_summary_with_two_baselines(MODEL_RUNS[2])


=== CVAE_20260424_164906_guide ===
Hyperparameters used
+-----------------------------+-----------------------------------------------------------------------------+
| Key                         | Value                                                                       |
+-----------------------------+-----------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_20260424_164906_guide/training_config_used.yaml        |
| model.type                  | cvae                                                                        |
| model.latent_dim            | 64                                                                          |
| model.encoder.type          | dictionary                                                                  |
| model.decoder.type          | dictionary                                                                  |
| model.prior.type            | learnable_dictionary           

In [23]:
# 4) CVAE Dictionary + Vamp Prior
result_model_4 = _print_model_summary_with_two_baselines(MODEL_RUNS[3])


=== CVAE_20260424_173354_guide_cnn ===
Hyperparameters used
+-----------------------------+-----------------------------------------------------------------------------+
| Key                         | Value                                                                       |
+-----------------------------+-----------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_20260424_173354_guide_cnn/training_config_used.yaml    |
| model.type                  | cvae                                                                        |
| model.latent_dim            | 64                                                                          |
| model.encoder.type          | dictionary_cnn                                                              |
| model.decoder.type          | dictionary_cnn                                                              |
| model.prior.type            | learnable_dictionary       

In [24]:
# 5) CVAE Gaussian
result_model_5 = _print_model_summary_with_two_baselines(MODEL_RUNS[4])


=== CVAE_20260424_174908_hierarcchy ===
Hyperparameters used
+-----------------------------+-----------------------------------------------------------------------------+
| Key                         | Value                                                                       |
+-----------------------------+-----------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_20260424_174908_hierarcchy/training_config_used.yaml   |
| model.type                  | cvae                                                                        |
| model.latent_dim            | 64                                                                          |
| model.encoder.type          | dictionary                                                                  |
| model.decoder.type          | dictionary                                                                  |
| model.prior.type            | learnable_dictionary      

In [25]:
# 6) Diffusion
result_model_6 = _print_model_summary_with_two_baselines(MODEL_RUNS[5])


=== CVAE_20260424_180137_vamp ===
Hyperparameters used
+-----------------------------+-----------------------------------------------------------------------------+
| Key                         | Value                                                                       |
+-----------------------------+-----------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_20260424_180137_vamp/training_config_used.yaml         |
| model.type                  | cvae                                                                        |
| model.latent_dim            | 64                                                                          |
| model.encoder.type          | dictionary                                                                  |
| model.decoder.type          | dictionary                                                                  |
| model.prior.type            | vampprior                       

#### 3-phase models

In [ ]:
# 1) AutoRegressiveTransformerVAE
result_model_1 = _print_model_summary_with_two_baselines(MODEL_RUNS[0])

In [26]:
# 7) SetGuideVAE (3-phase) - use aggregated phase eval artifacts
setguide_model_dir = HPC_RUNS_DIR / 'SetGuideVAE_DictionaryCoder_20260416_103152'
setguide_config = setguide_model_dir / 'training_config_used.yaml'
shared_eval_plan_setguide = _build_shared_eval_plan(
    data_variant='3phase_15min',
    max_files=None,
    seed=42,
    config_path=setguide_config if setguide_config.exists() else CONFIG_PATH,
    )

print('\n=== SetGuideVAE_DictionaryCoder_20260416_103152 ===')
_print_model_hyperparameters(setguide_model_dir)

setguide_agg_eval_json = setguide_model_dir / 'aggregated_phase_eval' / 'eval_results.json'
if setguide_agg_eval_json.exists():
    print('Using aggregated SetGuide evaluation results.')
    result_setguide = _display_eval_results(setguide_agg_eval_json, shared_plan=shared_eval_plan_setguide)
    _show_visual_artifacts(setguide_agg_eval_json.parent)
else:
    print('Aggregated eval not found, falling back to top-level SetGuide eval_results.json.')
    result_setguide = evaluate_model_run(
        model_name='SetGuideVAE_DictionaryCoder_20260416_103152',
        shared_plan=shared_eval_plan_setguide,
        rerun_if_supported=False,
        )

NameError: name '_build_shared_eval_plan' is not defined

### Overall Comparison Across All HPC Runs

This cell collects each model folder's `eval_results.json` and compares them side-by-side using `compare_eval`.

#### 1-phase

In [113]:
import importlib
import pandas as pd
import evaluation.compare_models as compare_models

importlib.reload(compare_models)

ModelComparator = compare_models.PhasedModelComparator

comparison_output_dir = PROJECT_ROOT / "evaluation_comparison_hpc_runs"
comparator = ModelComparator(output_dir=str(comparison_output_dir))

DISPLAY_MODEL_NAMES = {
    "AutoRegressiveTransformerVAE_20260424_170101": "ART-VAE",
    "CVAE_20260424_163736": "CVAE",
    "CVAE_20260424_164906_guide": "GUIDE-VAE",
    "CVAE_20260424_173354_guide_cnn": "GUIDE-VAE-CNN",
    "CVAE_20260424_174908_hierarcchy": "GUIDE-VAE-Hierarchy",
    "CVAE_20260424_180137_vamp": "GUIDE-VAE-Vamp",
    "DiffusionModel_20260424_171827": "Diffusion",
    "AverageBaselineModel_20260425_192621": "Avg Baseline",
    "QuantileBaselineModel_20260425_193656": "Quantile Baseline",
}

added = []

for model_name in MODEL_RUNS:
    model_dir = HPC_RUNS_DIR / model_name
    eval_json = model_dir / "aggregated_phase_eval" / "eval_results.json"

    if eval_json.exists():
        display_name = DISPLAY_MODEL_NAMES.get(model_name, model_name)

        # IMPORTANT: pass model_dir, not eval_json
        comparator.add_model(display_name, str(model_dir))

        added.append(display_name)
    else:
        print(f"Skipping {model_name}: eval results not found at {eval_json}")

if not added:
    print("No eval result files found. Run model cells first.")
else:
    print(f"Comparing {len(added)} models...")
    print("Legend names:", ", ".join(added))

    # Compute averages before making notebook table
    comparator._compute_averages("aggregated_phase")

    # Build readable notebook table
    all_metrics = sorted({
        metric
        for model_name in added
        for metric, value in comparator.models_data[model_name]["aggregated_phase"].items()
        if isinstance(value, (int, float))
    })

    # Put summary scores last
    preferred_last = [
        "Avg MinMax Synthetic Score",
        "Avg MinMax Reconstruction Score",
        "BEST overall synthetic score",
        "BEST overall reconstruction score",
    ]

    non_summary = [m for m in all_metrics if m not in preferred_last]
    summary = [m for m in preferred_last if m in all_metrics]
    all_metrics = non_summary + summary

    table_df = pd.DataFrame(index=all_metrics, columns=added, dtype=float)

    for model_name in added:
        metrics = comparator.models_data[model_name]["aggregated_phase"]
        for metric_name in all_metrics:
            metric_val = metrics.get(metric_name)
            if isinstance(metric_val, (int, float)):
                table_df.loc[metric_name, model_name] = metric_val

    table_df.index.name = "Metric"

    print("\nReadable model comparison table:")
    display(table_df.round(4))

    comparator.compare()

    print(f"Comparison artifacts written to: {comparison_output_dir}")

Comparing 9 models...
Legend names: ART-VAE, Avg Baseline, CVAE, GUIDE-VAE, GUIDE-VAE-CNN, GUIDE-VAE-Hierarchy, GUIDE-VAE-Vamp, Diffusion, Quantile Baseline

Readable model comparison table:


,ART-VAE,Avg Baseline,CVAE,GUIDE-VAE,GUIDE-VAE-CNN,GUIDE-VAE-Hierarchy,GUIDE-VAE-Vamp,Diffusion,Quantile Baseline
Metric,,,,,,,,,
ACF Distribution Match (KS),0.0833,0.8030,0.1921,0.0723,0.0528,0.0896,0.0715,0.0470,0.4747
Aggregated Profiles (Flexibility Distance),28.7740,71.0541,54.1626,56.6864,59.6773,46.6229,54.2097,45.4824,66.4442
Classical Statistics,0.1468,0.2936,0.1349,0.1689,0.1537,0.1598,0.1736,0.0929,0.0915
Flexibility Distance,0.6749,0.6086,0.4814,0.5123,0.4137,0.4321,0.4415,0.2896,0.5970
Per-Condition Evaluation,0.1011,0.5036,0.2526,0.0775,0.0743,0.0922,0.0860,0.0751,0.1838
Reconstruction Log-Likelihood,-1.8265,-1.6458,-1.4018,-1.5879,-1.4248,-1.4545,-1.4375,-0.9814,-1.6933
Reconstruction RMSE,1.5031,1.2537,0.9830,1.1840,1.0059,1.0362,1.0182,0.6399,1.3087
Set-MMD (Permutation Invariant),0.0185,0.2857,0.1332,0.0772,0.0641,0.0778,0.0816,0.0286,0.1411
"TSTR (Train on Synthetic, Test on Real)",1.1883,1.5000,1.2189,1.1953,1.2705,1.1750,1.2018,1.2266,1.2261



--- Comparing models for phase: aggregated_phase ---

Model Comparison Table (aggregated_phase):
[['FD on Real & Synth Aggregated Profiles', '28.7740', '71.0541', '54.1626', '56.6864', '59.6773', '46.6229', '54.2097', '45.4824', '66.4442'], ['Reconstruction Flexibility Distance (FD)', '0.6749', '0.6086', '0.4814', '0.5123', '0.4137', '0.4321', '0.4415', '0.2896', '0.5970'], ['Reconstruction Log-Likelihood', '-1.8265', '-1.6458', '-1.4018', '-1.5879', '-1.4248', '-1.4545', '-1.4375', '-0.9814', '-1.6933'], ['Reconstruction RMSE', '1.5031', '1.2537', '0.9830', '1.1840', '1.0059', '1.0362', '1.0182', '0.6399', '1.3087'], ['ACF Distribution Match (KS)', '0.0833', '0.8030', '0.1921', '0.0723', '0.0528', '0.0896', '0.0715', '0.0470', '0.4747'], ['Per-Condition Evaluation', '0.1011', '0.5036', '0.2526', '0.0775', '0.0743', '0.0922', '0.0860', '0.0751', '0.1838'], ['Maximum Mean Discrepancy (MMD)', '0.0185', '0.2857', '0.1332', '0.0772', '0.0641', '0.0778', '0.0816', '0.0286', '0.1411'], ['Tr

In [ ]:
for model_name in comparator.models_data:
    v = comparator.models_data[model_name]["aggregated_phase"].get("Avg MinMax Synthetic Score")
    print(model_name, v)

#### 3-phase

In [91]:
PRIMARY_HIERARCHY_MODEL = PROJECT_ROOT / "models" / "hpc_runs" / "CVAE_20260424_174908_hierarcchy"
SECONDARY_SPLITTER_MODEL = PROJECT_ROOT / "models" / "hpc_runs" / "3-phase" / "HierarchicalSplitter_20260426_182151"

MODEL_RUNS_TO_EVAL = [
    "SetGuideVAE_20260426_180021",
    "DiffusionModel_20260426_182935",
    "AverageBaselineModel_20260426_185808",
    "QuantileBaselineModel_20260426_190605",
]

model_paths = [
    str(HPC_RUNS_DIR_PHASED / name)
    for name in MODEL_RUNS_TO_EVAL
]

model_names = [
    "SetGuideVAE",
    "Diffusion",
    "Avg Baseline",
    "Quantile Baseline",
]

secondary_model_paths = [None] * len(model_paths)

# Add hierarchy model: primary = aggregate CVAE, secondary = splitter
model_paths.append(str(PRIMARY_HIERARCHY_MODEL))
model_names.append("GUIDE-VAE-Hierarchy")
secondary_model_paths.append(str(SECONDARY_SPLITTER_MODEL))

In [92]:
secondary_model_paths = []

for model_name in MODEL_RUNS_TO_EVAL:
    if model_name == "HierarchicalSplitter_20260426_182151":
        secondary_model_paths.append(
            str(HPC_RUNS_DIR_PHASED / "HierarchicalSplitter_20260426_182151")
        )
    else:
        secondary_model_paths.append(None)

In [96]:
results = compare_models.compare_models(
    model_paths=model_paths,
    model_names=model_names,
    secondary_model_paths=secondary_model_paths,
    data_dir="/Volumes/USB/raw_profiles",
    num_samples=100,
    output_dir=str(comparison_output_dir),
    phased=True,
    recompute=True,
)


Evaluating model: SetGuideVAE


AttributeError: module 'torch.mps' has no attribute 'current_device'

In [ ]:
import importlib
import pandas as pd
import evaluation.compare_models as compare_models

importlib.reload(compare_models)

ModelComparator = compare_models.PhasedModelComparator

comparison_output_dir = PROJECT_ROOT / "evaluation_comparison_hpc_runs_3phase"
comparator = ModelComparator(output_dir=str(comparison_output_dir))

DISPLAY_MODEL_NAMES = {
    "AutoRegressiveTransformerVAE_20260426_184212": "ART-VAE",
    "AverageBaselineModel_20260426_185808": "Avg Baseline",
    "DiffusionModel_20260426_182935": "Diffusion",
    "HierarchicalSplitter_20260426_182151": "Hierarchical Splitter",
    "QuantileBaselineModel_20260426_190605": "Quantile Baseline",
    "SetGuideVAE_20260426_180021": "SetGuideVAE",
}

added = []

for model_name in MODEL_RUNS_PHASED:
    model_dir = HPC_RUNS_DIR_PHASED / model_name
    eval_json = model_dir / "aggregated_phase_eval" / "eval_results.json"

    if eval_json.exists():
        display_name = DISPLAY_MODEL_NAMES.get(model_name, model_name)
        comparator.add_model(display_name, str(model_dir))
        added.append(display_name)
    else:
        print(f"Skipping {model_name}: eval results not found at {eval_json}")


if not added:
    print("No eval result files found. Run model cells first.")
else:
    print(f"Comparing {len(added)} models...")
    print("Legend names:", ", ".join(added))

    # Compute averages before making notebook table
    comparator._compute_averages("aggregated_phase")

    # Build readable notebook table
    all_metrics = sorted({
        metric
        for model_name in added
        for metric, value in comparator.models_data[model_name]["aggregated_phase"].items()
        if isinstance(value, (int, float))
    })

    # Put summary scores last
    preferred_last = [
        "Avg MinMax Synthetic Score",
        "Avg MinMax Reconstruction Score",
        "BEST overall synthetic score",
        "BEST overall reconstruction score",
    ]

    non_summary = [m for m in all_metrics if m not in preferred_last]
    summary = [m for m in preferred_last if m in all_metrics]
    all_metrics = non_summary + summary

    table_df = pd.DataFrame(index=all_metrics, columns=added, dtype=float)

    for model_name in added:
        metrics = comparator.models_data[model_name]["aggregated_phase"]
        for metric_name in all_metrics:
            metric_val = metrics.get(metric_name)
            if isinstance(metric_val, (int, float)):
                table_df.loc[metric_name, model_name] = metric_val

    table_df.index.name = "Metric"

    print("\nReadable model comparison table:")
    display(table_df.round(4))

    comparator.compare()

    print(f"Comparison artifacts written to: {comparison_output_dir}")

Comparing 5 models...
Legend names: ART-VAE, Avg Baseline, Diffusion, Quantile Baseline, SetGuideVAE

Readable model comparison table:


,ART-VAE,Avg Baseline,Diffusion,Quantile Baseline,SetGuideVAE
Metric,,,,,
ACF Distribution Match (KS),0.1897,0.8923,0.2654,0.3622,0.2467
Aggregated Profiles (Flexibility Distance),25.2479,68.0800,59.0374,46.1820,50.1161
Classical Statistics,0.2177,0.1745,0.2560,0.2195,0.2303
Flexibility Distance,0.7830,0.6360,0.7807,0.6185,0.7837
Per-Condition Evaluation,0.1623,0.5374,0.1694,0.1663,0.1971
Reconstruction Log-Likelihood,-1.8175,-1.5896,-1.9324,-1.6839,-1.9479
Reconstruction RMSE,1.4894,1.1859,1.6708,1.2907,1.6964
Set-MMD (Permutation Invariant),0.0598,0.2148,0.0913,0.1029,0.0806
"TSTR (Train on Synthetic, Test on Real)",0.5895,0.5933,0.5920,0.6124,0.5678



--- Comparing models for phase: aggregated_phase ---

Model Comparison Table (aggregated_phase):
[['Reconstruction Aggregated Profiles (FD)', '25.2479', '68.0800', '59.0374', '46.1820', '50.1161'], ['Reconstruction Flexibility Distance (FD)', '0.7830', '0.6360', '0.7807', '0.6185', '0.7837'], ['Reconstruction Log-Likelihood', '-1.8175', '-1.5896', '-1.9324', '-1.6839', '-1.9479'], ['Reconstruction RMSE', '1.4894', '1.1859', '1.6708', '1.2907', '1.6964'], ['ACF Distribution Match (KS)', '0.1897', '0.8923', '0.2654', '0.3622', '0.2467'], ['Per-Condition Evaluation', '0.1623', '0.5374', '0.1694', '0.1663', '0.1971'], ['Maximum Mean Discrepancy (MMD)', '0.0598', '0.2148', '0.0913', '0.1029', '0.0806'], ['Train on Synthetic, Test on Real (TSTR)', '0.5895', '0.5933', '0.5920', '0.6124', '0.5678'], ['BEST Overall Reconstruction Score', '0.4434', '0.7235', '0.0806', '0.7607', '0.1049'], ['BEST Overall Synthetic Score', '1.0000', '0.0000', '0.8444', '0.7380', '0.8922']]

Table saved to /Users/